In [1]:
import heapq
import time
goal_state = (1,2,3,
              4,5,6,
              7,8,0)#global variable so we dont have to declare it again and again

In [2]:
class TreeNode:
    def __init__(self, board, parent=None, g=0, h=0):
        self.board = board
        self.parent = parent
        self.g = g
        self.h = h
        self.f = g+h #total distance calculation for A*
    def __lt__(self, other):
        return self.f <other.f

In [3]:
def man_distance(board):
    dis = 0
    for i in range(9):
        value = board[i]
        if value!=0:
            goal_index = goal_state.index(value)

            cur_row = i//3
            cur_col = i%3#positions of the number

            goal_row = goal_index//3
            goal_col = goal_index%3
            dis+=abs(cur_row-goal_row)+abs(cur_col-goal_col)
    return dis

In [4]:
def getting_neighbor(board):
    neighbors = []
    index_0 = board.index(0)
    row = index_0//3
    col = index_0%3

    moves = [(-1,0),#moving up
             (1, 0),#down
             (0, -1),#left
             (0, 1)]#rigght

    for move in moves:
        new_row = row+move[0]
        new_col = col+move[1]

        if 0 <= new_row < 3 and 0 <= new_col < 3:
            new_index = new_row * 3 + new_col

            temp_board = list(board)#we need to modify hte board so converting from tuple to list

            temp_board[index_0], temp_board[new_index] = temp_board[new_index], temp_board[index_0]

            neighbors.append(tuple(temp_board))

    return neighbors

In [5]:
def reconcstruct(node):#reconstuct the path to thte solution
    path = []
    while node:
        path.append(node.board)
        node = node.parent
    path.reverse()#since we are going backwards to reconstruct, we need to reverse to actially get the solution
    return path

In [9]:
# def a_star_search(start_board):
#     frontier = list()
#     explored = set()#set of visited paths to avoid infinte looping
#
#     start_node = TreeNode(
#         board=start_board,
#         parent=None,
#
#         g=0,
#         h=man_distance(start_board)
#     )
#
#     heapq.heappush(frontier, start_node)
#
#     while frontier:
#         current_node = heapq.heappop(frontier)
#         if current_node.board == goal_state:
#             return reconcstruct(current_node)
#         explored.add(current_node.board)
#
#         for neighbor in getting_neighbor(current_node.board):
#             if neighbor in explored:
#                 continue
#             g_cost = current_node.g+1
#             h_cost = man_distance(neighbor)
#
#             neighbor_node = TreeNode(
#                 board=neighbor,
#                 parent=current_node,
#                 g=g_cost,
#                 h=h_cost
#             )
#
#             heapq.heappush(frontier, neighbor_node)
#     return None

def a_star_search(start_board, goal_board):
    # frontier stores tuples (f_score, path)
    frontier = []
    heapq.heappush(frontier, (man_distance(start_board), [start_board]))

    explored = set()

    while frontier:
        _, cur_path = heapq.heappop(frontier)
        cur_board = cur_path[-1]

        # check if goal reached
        if cur_board == goal_board:
            return cur_path

        explored.add(tuple(cur_board))  # make hashable if board is a list of lists

        for neighbor in getting_neighbor(cur_board):
            neighbor_tuple = tuple(neighbor)
            if neighbor_tuple in explored:
                continue

            new_path = cur_path + [neighbor]
            f_score = len(new_path) - 1 + man_distance(neighbor)  # g + h
            heapq.heappush(frontier, (f_score, new_path))

    return None
#user can go back to the previous state therefore this is a graph, there can be loops

In [7]:
def is_solvable(board):
    count = 0
    numbers = [num for num in board if num!=0]

    for i in range(len(numbers)):
        for j in range(i+1, len(numbers)):
            if numbers[i]>numbers[j]:
                count+=1
    return count%2==0#if number of inversions are even, it is solvable

In [8]:
def present(board):
    for i in range(0,9,3):
        print(board[i:i+3])
    print()#structre of the board

In [14]:
def testing(board):
    print("initial case;")
    present(board)
    if not is_solvable(board):
        print("This case isnt solvable")
        return
    solution = a_star_search(board, goal_state)
    if solution:
        print(f"{len(solution)-1} moves required")
        for step in solution:
            present(step)
    else:
        print("Not solvable")

In [19]:
def test_cases():
    return [
        (1,2,3,4,5,6,7,8,0),
        (1,2,3,4,5,6,0,7,8),#2solvable
        (0,1,3,4,2,5,7,8,6),
        (1,2,3,4,6,5,7,8,0),
        (8,1,2,0,4,3,7,6,5),#2unsolvalble
        (1,2,4,5,3,6,0,7,8),
        (1,2,3,4,5,7,6,8,0)
    ]

In [20]:
if __name__=="__main__":
    test_case = test_cases()
    for board in test_case:
        testing(board)

initial case;
(1, 2, 3)
(4, 5, 6)
(7, 8, 0)

0 moves required
(1, 2, 3)
(4, 5, 6)
(7, 8, 0)

initial case;
(1, 2, 3)
(4, 5, 6)
(0, 7, 8)

2 moves required
(1, 2, 3)
(4, 5, 6)
(0, 7, 8)

(1, 2, 3)
(4, 5, 6)
(7, 0, 8)

(1, 2, 3)
(4, 5, 6)
(7, 8, 0)

initial case;
(0, 1, 3)
(4, 2, 5)
(7, 8, 6)

4 moves required
(0, 1, 3)
(4, 2, 5)
(7, 8, 6)

(1, 0, 3)
(4, 2, 5)
(7, 8, 6)

(1, 2, 3)
(4, 0, 5)
(7, 8, 6)

(1, 2, 3)
(4, 5, 0)
(7, 8, 6)

(1, 2, 3)
(4, 5, 6)
(7, 8, 0)

initial case;
(1, 2, 3)
(4, 6, 5)
(7, 8, 0)

This case isnt solvable
initial case;
(8, 1, 2)
(0, 4, 3)
(7, 6, 5)

This case isnt solvable
initial case;
(1, 2, 4)
(5, 3, 6)
(0, 7, 8)

16 moves required
(1, 2, 4)
(5, 3, 6)
(0, 7, 8)

(1, 2, 4)
(5, 3, 6)
(7, 0, 8)

(1, 2, 4)
(5, 3, 6)
(7, 8, 0)

(1, 2, 4)
(5, 3, 0)
(7, 8, 6)

(1, 2, 4)
(5, 0, 3)
(7, 8, 6)

(1, 2, 4)
(0, 5, 3)
(7, 8, 6)

(0, 2, 4)
(1, 5, 3)
(7, 8, 6)

(2, 0, 4)
(1, 5, 3)
(7, 8, 6)

(2, 4, 0)
(1, 5, 3)
(7, 8, 6)

(2, 4, 3)
(1, 5, 0)
(7, 8, 6)

(2, 4, 3)
(1, 0, 5)
(7, 